In [ ]:
import numpy as np
import pandas as pd
import random
import torch
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
import os

In [ ]:
# GPU ?? ?? (Kaggle ???? GPU ?? ?)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Seed ??
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# ??? ???? (Kaggle ??)
#data_path = "/kaggle/input/dataset-enterprise"
#train = pd.read_csv(f"{data_path}/train.csv")
#test = pd.read_csv(f"{data_path}/test.csv")
#train = pd.read_csv(f"{data_path}/train.csv")
#test = pd.read_csv(f"{data_path}/test.csv")
#sample_submission = pd.read_csv(f"{data_path}/sample_submission.csv")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [ ]:
train = train.drop(columns=['ID'], axis=1)
test = test.drop(columns=['ID'], axis=1)

In [ ]:
#???? ???
def parse_value(x):
    if pd.isnull(x): return 0
    elif '6000' in str(x): return 6
    elif '4500' in str(x): return 5
    elif '3500' in str(x): return 4
    elif '2500' in str(x): return 3
    elif '1500' in str(x): return 2
    else: return 1

#?? NaN? ??
def fit_label_encoder(train_series, test_series):
    train_series = train_series.fillna('Missing').astype(str)
    test_series = test_series.fillna('Missing').astype(str)

    le = LabelEncoder()
    full_data = pd.concat([train_series, test_series])
    le.fit(full_data)

    return le, le.transform(train_series), le.transform(test_series)

In [ ]:
# ???? ?????? ??? ??
CURRENT_YEAR = 2025
train['??'] = CURRENT_YEAR - train['????']
test['??'] = CURRENT_YEAR - test['????']

# # ??? ??? ??
# numeric_features = ['?? ?','???(???)','? ???(??)','???(??)','SNS ??? ?(???)']
# for feature in numeric_features:
#     mean_value = train[feature].mean()
#     train[feature] = train[feature].fillna(mean_value)
#     test[feature] = test[feature].fillna(mean_value)

# 1. KNNImputer ?? (?????)
from sklearn.impute import KNNImputer
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('????')  # ?? ??

imputer = KNNImputer(n_neighbors=5)
train_num = pd.DataFrame(imputer.fit_transform(train[numeric_cols]), columns=numeric_cols)
test_num = pd.DataFrame(imputer.transform(test[numeric_cols]), columns=numeric_cols)

# ?? ???/??? ??? ???? ?? ??
train_cat = train.drop(columns=numeric_cols + ['????'])
test_cat = test.drop(columns=numeric_cols)

# ???/??? ?? ???
train = pd.concat([train_cat.reset_index(drop=True), train_num.reset_index(drop=True), train['????'].reset_index(drop=True)], axis=1)
test = pd.concat([test_cat.reset_index(drop=True), test_num.reset_index(drop=True)], axis=1)


# ?? ???
le_country, train['??'], test['??'] = fit_label_encoder(train['??'], test['??'])
le_field, train['??'], test['??'] = fit_label_encoder(train['??'], test['??'])

# ??? ??
stage_mapping = {'Seed': 0, 'Series A': 1, 'Series B': 2, 'Series C': 3, 'IPO': 4}
train['????'] = train['????'].map(stage_mapping).fillna(0)
test['????'] = test['????'].map(stage_mapping).fillna(0)

# ???? ???

train['????_???'] = train['????(???)'].apply(parse_value)
test['????_???'] = test['????(???)'].apply(parse_value)

# ??? ??? ???? ???
train['log_???'] = np.log1p(train['???(??)'])
train['log_???'] = np.log1p(train['? ???(??)'])
test['log_???'] = np.log1p(test['???(??)'])
test['log_???'] = np.log1p(test['? ???(??)'])

# xx?? ??? ??
bool_map = {'Yes': 1, 'No': 0}
for feature in ['????', '????']:
    train[feature] = train[feature].map(bool_map).fillna(0)
    test[feature] = test[feature].map(bool_map).fillna(0)

# SNS ???
def sns_bin(x):
    if pd.isna(x): return 0
    if x < 1: return 0
    elif x < 3: return 1
    elif x < 5: return 2
    else: return 3
train['SNS?????'] = train['SNS ??? ?(???)'].apply(sns_bin)
test['SNS?????'] = test['SNS ??? ?(???)'].apply(sns_bin)


In [ ]:
# ???? ??
train['??????'] = train['log_???'] / (train['log_???'] + 1)
test['??????'] = test['log_???'] / (test['log_???'] + 1)

train['??????'] = train['log_???'] / (train['?? ?'] + 1)
test['??????'] = test['log_???'] / (test['?? ?'] + 1)

train['?????'] = train['log_???'] / (train['???(???)'] + 1)
test['?????'] = test['log_???'] / (test['???(???)'] + 1)

train['????????'] = train['log_???'] / (train['????_???'] + 1)
test['????????'] = test['log_???'] / (test['????_???'] + 1)

train['?????'] = train['log_???'] / (train['??'] + 1)
test['?????'] = test['log_???'] / (test['??'] + 1)

train['??????'] = train['????_???'] / (train['????'] + 1)
test['??????'] = test['????_???'] / (test['????'] + 1)

train['??????'] = train['log_???'] / (train['??'] + 1)
test['??????'] = test['log_???'] / (test['??'] + 1)

train['??????'] = train['???(???)'] / (train['??'] + 1)
test['??????'] = test['???(???)'] / (test['??'] + 1)

train['SNS????'] = train['SNS ??? ?(???)'] / (train['log_???'] + 1)
test['SNS????'] = test['SNS ??? ?(???)'] / (test['log_???'] + 1)

In [ ]:
features = [
    '??', '??', '??', '????', '?? ?', '???(???)', '? ???(??)', '???(??)',
    'SNS?????', '????_???', '????', '????', 
    '??????', '??????', '?????',
    '????????', '?????',
    '??????', '??????', '??????', 'SNS????'
]

cat_features = ['??', '??', '????']
cat_idxs = [features.index(col) for col in cat_features]
cat_dims = [
    len(le_country.classes_),
    len(le_field.classes_),
    train['????'].nunique()  # ?? len(train['????'].unique())?? OK
]

In [ ]:
# # ?? ??
# X = train[features].values
# y = train['????'].values.reshape(-1, 1)
# X_test = test[features].values

# # TabNet ?? ?? (GPU ??)
# from sklearn.metrics import mean_absolute_error
# kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
# cv_scores = []
# tabnet_preds = []

# for train_idx, valid_idx in kf.split(X):
#     model = TabNetRegressor(
#     n_d=32, n_a=32, n_steps=5, gamma=1.5, seed=SEED+10,
#     cat_idxs=cat_idxs,
#     cat_dims=cat_dims,
#     device_name=device
#     )
#     model.fit(
#         X[train_idx], y[train_idx],
#         eval_set=[(X[valid_idx], y[valid_idx])],
#         max_epochs=200,
#         patience=10,
#         batch_size=512,
#         virtual_batch_size=128,
#         eval_metric=['mae'],
#         #verbose=0
#     )
#     val_preds = model.predict(X[valid_idx]).ravel()
#     fold_mae = mean_absolute_error(y[valid_idx].ravel(), val_preds)
#     cv_scores.append(fold_mae)
#     tabnet_preds.append(model.predict(X_test))
        
#     # LightGBM
#     lgb_model = lgb.LGBMRegressor(random_state=SEED)
#     lgb_model.fit(X, y.ravel())
#     lgb_preds = lgb_model.predict(X_test)

#     # GradientBoosting
#     gbr = GradientBoostingRegressor(random_state=SEED)
#     gbr.fit(X, y.ravel())
#     gbr_preds = gbr.predict(X_test)

# tabnet_final = np.mean(tabnet_preds, axis=0)
# lgb_final = np.mean(lgb_preds, axis=0)
# gbr_final = np.mean(gbr_preds, axis=0)

# # ???
# final_preds = (tabnet_final.ravel() + lgb_preds + gbr_preds) / 3
# estimated_lb_score = np.mean(np.abs(final_preds - y.mean()))
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
# ??? ??
X = train[features]
y = train['????']
X_test = test[features]
# KFold ??
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []
models = []
test_preds = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[valid_idx], y.iloc[valid_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=15,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    cv_scores.append(mae)
    models.append(model)

    # ???? ?? ?? (optional)
    test_preds.append(model.predict(X_test))
    print(f"Fold {fold+1} MAE: {mae:.4f}")

print(f"\nAverage MAE across folds: {np.mean(cv_scores):.5f}")

# ??? ?? ?? (optional)
final_test_pred = np.mean(test_preds, axis=0)



In [ ]:
test_preds = np.zeros(len(X_test))

for model in models:
    test_preds += model.predict(X_test) / kf.get_n_splits()

final_preds = test_preds

print("\n? Test ?? ?? (KFold ?? ??)")
print(final_preds[:10])

In [ ]:
assert len(test) == len(final_preds), "???? ???? ??? ???? ????."

sample_submission['????'] = final_preds
sample_submission.to_csv('./submission_xgb_kfold.csv', index=False)

print("? ?? ?? ?? ??: submission_xgb_kfold.csv")
sample_submission.head()